# Capítulo 24 · Laboratório 24: The Strat
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Regras mecânicas e simplificadas, escritas para o teste: não reproduzem a leitura discricionária de Rob Smith. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap24.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O Yahoo só guarda ~730 pregões de barras de 60 minutos, contados de hoje. As barras do livro vão de 23/10/2024 a 30/09/2026: quando esse período sai da janela, o notebook usa o que o Yahoo ainda guarda e avisa, e as datas e os números das Partes 7 e 8 mudam. O laboratório conta os três cenários, classifica as viradas nas quatro reversões, testa o próximo 2 depois de uma barra 3, os combos no diário e no 60 minutos (com a continuidade temporal como filtro), a magnitude e o efeito do relógio.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
INI60 = "2024-10-23"                    # início das barras de 60 minutos do livro
CACHE = "dados_cap24.pkl"


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


pc = lambda x, k=0: "" if pd.isna(x) else f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=2: "" if pd.isna(x) else ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:+,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
nn = lambda x: f"{x:,}".replace(",", ".")
B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
INTRA = ["SPY", "QQQ", "IWM", "DIA", "BOVA11.SA", "PETR4.SA", "VALE3.SA", "ITUB4.SA"]

## Parte 1: dados

In [ ]:
if os.path.exists(CACHE):
    IDX, D, H60 = pd.read_pickle(CACHE)
else:
    # O Yahoo só aceita data de início nos últimos ~730 dias (60 minutos). Pedindo o período máximo (period), ele
    # devolve ~730 pregões; o corte refaz o período do livro enquanto ele couber.
    def intra(tickers, periodo, intervalo, ini):
        x = yf.download(tickers, period=periodo, interval=intervalo, auto_adjust=True, progress=False,
                        group_by="ticker")
        confere({f"as barras de {intervalo[:-1]} minutos": x})
        dia = x.index.strftime("%Y-%m-%d")
        corte = x[(dia >= ini) & (dia < FIM)]
        return corte if len(corte) else x       # o período do livro saiu da janela: fica a mais recente

    IDX = yf.download(["^BVSP", "^GSPC"], start="2000-01-01", end=FIM, auto_adjust=True, progress=False,
                      group_by="ticker")
    D = yf.download(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, progress=False, group_by="ticker")
    H60 = intra(INTRA, "730d", "60m", INI60)
    confere({"o Ibovespa e o S&P 500": IDX, "as 40 ações": D})
    if f"{H60.index[0]:%Y-%m-%d}" > "2024-10-23" or f"{H60.index[-1]:%Y-%m-%d}" != "2026-09-30":
        print(f"Aviso: o Yahoo só guarda ~730 pregões de barras de 60 minutos, contados de hoje. Esta execução usa a janela"
              f" que o Yahoo guarda hoje, e não a do livro (23/10/2024 a 30/09/2026): de {H60.index[0]:%d/%m/%Y} a"
              f" {H60.index[-1]:%d/%m/%Y}. As datas e os números das Partes 7 e 8 diferem dos do livro.")
    pd.to_pickle((IDX, D, H60), CACHE)
NOMES = {"^BVSP": "Ibovespa", "^GSPC": "S&P 500"}


def limpa(d):
    """Sem filtro de volume: o Yahoo informa volume zero em algumas barras intradiárias da B3 (a das 10h)."""
    d = d.drop(columns=["Adj Close"], errors="ignore").dropna()
    return d[(d["Close"] > 0) & (d["High"] > d["Low"])]


def agrega(d, regra):
    return d.resample(regra).agg({"Open": "first", "High": "max", "Low": "min", "Close": "last"}).dropna()


def cenario(d):
    """1 = inside; 2u / 2d = direcional para cima / para baixo; 3 = outside. A primeira barra fica sem cenário."""
    h, l = d["High"], d["Low"]
    acima, abaixo = h > h.shift(), l < l.shift()
    c = np.select([acima & abaixo, acima, abaixo], ["3", "2u", "2d"], "1")
    c = pd.Series(c, index=d.index)
    c.iloc[0] = ""
    return c


def zigzag(c, limiar=0.05):
    """Pivôs: um topo (ou fundo) só é confirmado depois de uma reversão de pelo menos 'limiar'."""
    piv, tend, ext_i, ext_p = [], 0, 0, c[0]
    for i, p in enumerate(c):
        if tend >= 0 and p > ext_p or tend < 0 and p < ext_p:
            ext_i, ext_p = i, p
        if tend >= 0 and p <= ext_p * (1 - limiar):
            piv.append((ext_i, "topo")); tend, ext_i, ext_p = -1, i, p
        elif tend <= 0 and p >= ext_p * (1 + limiar):
            piv.append((ext_i, "fundo")); tend, ext_i, ext_p = 1, i, p
    return piv


IX = {NOMES[t]: limpa(IDX[t]) for t in NOMES}

## Parte 2: os três cenários por tempo gráfico

In [ ]:
print("Parte 2. Os três cenários por tempo gráfico (desde 2000)")
print("   índice      tempo       1 (inside)   2 (direcional)   3 (outside)   2 que fechou contra")
for nome, d in IX.items():
    for tf, regra in [("diário", None), ("semanal", "W-FRI"), ("mensal", "ME")]:
        x = d if regra is None else agrega(d, regra)
        cen = cenario(x).iloc[1:]
        s = cen.str[0].value_counts(normalize=True)
        x = x.iloc[1:]
        falhou = ((cen == "2u") & (x["Close"] < x["Open"])) | ((cen == "2d") & (x["Close"] > x["Open"]))
        print(f"   {nome:<10} {tf:<9} {pc(s.get('1', 0)):>10} {pc(s.get('2', 0)):>15} {pc(s.get('3', 0)):>13}"
              f" {pc(falhou.sum() / (cen.str[0] == '2').sum()):>17}")
print("   '2 que fechou contra': barra 2 para cima que fechou abaixo da abertura (ou 2 para baixo que fechou acima).")

## Parte 3: as quatro reversões nos pivôs reais

In [ ]:
print("\nParte 3. Pivôs do zigue-zague de 5% no diário (os dois índices): como aconteceu a virada?")
tipos = []
for nome, d in IX.items():
    cen = cenario(d).to_numpy()
    for i, tipo in zigzag(d["Close"].to_numpy()):
        if i < 1 or i + 2 >= len(cen):
            continue
        b0, b1, b2 = cen[i], cen[i + 1], cen[i + 2]            # barra do pivô e as duas seguintes
        volta = "2u" if tipo == "fundo" else "2d"
        if b0 == "3" and b1 == "1" and b2 == volta:
            tipos.append("3-1-2")
        elif b1 == "3" or b0 == "3":
            tipos.append("barra 3 na virada")
        elif b1 == "1" and b2 == volta:
            tipos.append("2-1-2")
        elif b1 == volta:
            tipos.append("2-2")
        else:
            tipos.append("virada mais lenta (mais de três barras)")
t = pd.Series(tipos).value_counts(normalize=True)
print(f"   {nn(len(tipos))} pivôs")
for k, v in t.items():
    print(f"   {k:<42} {pc(v)}")

## Parte 4: depois de uma barra 3, onde está o próximo 2?

In [ ]:
print("\nParte 4. Depois de uma barra 3, para que lado sai o próximo 2? (2 índices e 40 ações)")
SERIES = dict(IX)
for tk in TICKERS:
    x = limpa(D[tk])
    if len(x) > 500:
        SERIES[tk] = x
for tf, regra in [("diário", None), ("semanal", "W-FRI"), ("mensal", "ME")]:
    lado = []
    for nome, d in SERIES.items():
        x = d if regra is None else agrega(d, regra)
        cen = cenario(x).to_numpy()
        H, L, C = x["High"].to_numpy(), x["Low"].to_numpy(), x["Close"].to_numpy()
        for i in range(1, len(cen) - 1):
            pos = (C[i] - L[i]) / (H[i] - L[i])                   # onde a barra fechou dentro dela (0 = mínima)
            k = i + 1                                             # o próximo 2 em relação a esta barra:
            while k < len(cen) and H[k] <= H[i] and L[k] >= L[i]:  # pula as inside bars
                k += 1
            if k >= len(cen):
                continue
            sobe, desce = H[k] > H[i], L[k] < L[i]
            lado.append((cen[i] == "3", pos, "ambos" if sobe and desce else ("cima" if sobe else "baixo")))
    A4 = pd.DataFrame(lado, columns=["tres", "pos", "lado"])
    for rot, a4 in [("barras 3", A4[A4["tres"]]), ("todas", A4)]:
        u = a4[a4["lado"] != "ambos"]
        alto, baixo = u[u["pos"] >= 2 / 3], u[u["pos"] <= 1 / 3]
        print(f"   {tf:<8} {rot:<8} {nn(len(a4)):>7} | sai por cima se fechou no terço de cima: {pc((alto['lado'] == 'cima').mean())};"
              f" no terço de baixo: {pc((baixo['lado'] == 'cima').mean())} | rompe os dois lados juntos: {pc((a4['lado'] == 'ambos').mean())}")
print("   Comparação com 'todas': boa parte do efeito é geometria (quem fecha perto da máxima está mais perto dela).")

## Parte 5: os combos no diário

In [ ]:
def resultado(H, L, C, i, entrada, stop, alvo, prazo=20, otimista=False):
    """Resultado em R de uma compra feita na barra i. Quando a mesma barra toca alvo e stop, não se sabe a ordem:
    o modo conservador conta perda; o otimista, ganho. Na própria barra de entrada, o otimista supõe que a mínima
    veio antes do gatilho (o stop não foi tocado depois da entrada), a menos que a barra tenha fechado no stop ou abaixo.
    A verdade fica entre os dois."""
    risco = entrada - stop
    if risco <= 0 or alvo <= entrada:
        return np.nan
    if otimista and C[i] <= stop and H[i] < alvo:
        return -1.0          # fechou no stop ou abaixo sem tocar o alvo: o stop veio depois da entrada
    fim = min(i + prazo, len(C))
    for k in range(i, fim):
        bate_stop, bate_alvo = L[k] <= stop, H[k] >= alvo
        if otimista and k == i:
            bate_stop = False
        if bate_stop and bate_alvo:
            return (alvo - entrada) / risco if otimista else -1.0
        if bate_stop:
            return -1.0
        if bate_alvo:
            return (alvo - entrada) / risco
    return (C[fim - 1] - entrada) / risco


def combos(d, prazo=20):
    """Combos de alta do Strat numa série. Gatilho: romper a máxima da barra anterior (se a barra abrir acima dele,
    a entrada é na abertura). Stop: mínima da barra anterior. Alvo: a máxima anterior, conforme a figura dos combos."""
    cen = cenario(d).to_numpy()
    O, H, L, C = (d[k].to_numpy() for k in ["Open", "High", "Low", "Close"])
    tr = pd.concat([d["High"] - d["Low"], (d["High"] - d["Close"].shift()).abs(),
                    (d["Low"] - d["Close"].shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(14).mean().to_numpy()
    out = []
    for i in range(4, len(d)):
        a, b, c_ = cen[i - 2], cen[i - 1], cen[i]
        if c_ not in ("2u", "3"):       # o gatilho é a máxima anterior; se a barra virar 3, ela também furou a mínima
            continue
        entrada, stop = max(H[i - 1], O[i]), L[i - 1]
        if b == "1" and a == "2u":
            nome, alvo = "2-1-2 de continuação", H[i - 2]
        elif b == "1" and a == "2d":
            nome, alvo = "2-1-2 de reversão", H[i - 2]
        elif b == "1" and a == "3":
            nome, alvo = "3-1-2 de reversão", H[i - 2]
        elif b == "2d" and a == "1":
            nome, alvo = "1-2-2 RevStrat", H[i - 2]                 # a máxima mais próxima: a da barra de dentro
        elif b == "2d" and a == "3":
            nome, alvo = "3-2-2 de reversão", H[i - 2]
        elif b == "2d":
            nome, alvo = "2-2 de reversão", H[i - 2]
        elif b == "2u":
            nome, alvo = "2-2 de continuação", H[i - 1] + (H[i - 1] - L[i - 1])
        else:
            continue
        if not H[i - 1] - L[i - 1] >= 0.25 * atr[i - 1]:     # barra de sinal minúscula: risco quase zero distorce o R
            continue
        alvo = alvo if alvo > entrada else entrada + (entrada - stop)      # sem máxima acima: alvo de 1R
        out.append({"i": i, "data": d.index[i], "combo": nome, "gap": O[i] > H[i - 1],
                    "r": resultado(H, L, C, i, entrada, stop, alvo, prazo),
                    "r_otim": resultado(H, L, C, i, entrada, stop, alvo, prazo, otimista=True),
                    "dist": (alvo - entrada) / (entrada - stop)})
    return pd.DataFrame(out)


ORDEM = ["2-1-2 de continuação", "2-1-2 de reversão", "3-1-2 de reversão", "1-2-2 RevStrat", "3-2-2 de reversão",
         "2-2 de reversão", "2-2 de continuação"]


def tabela(R, titulo):
    print(titulo)
    print("   combo                      casos   resultado médio (conserv. | otim.)   acerto (conserv. | otim.)   alvo mediano")
    for k in ORDEM:
        x = R[R["combo"] == k]
        if len(x) == 0:
            continue
        print(f"   {k:<24} {nn(len(x)):>8}   {nr(x['r'].mean()):>7} R | {nr(x['r_otim'].mean()):>6} R"
              f"            {pc((x['r'] > 0).mean()):>5} | {pc((x['r_otim'] > 0).mean()):>4}          {nr(x['dist'].median())[1:]} R")


CD = pd.concat([combos(SERIES[tk]).assign(ativo=tk) for tk in TICKERS if tk in SERIES])
CD = CD.dropna(subset=["r", "r_otim"])
tabela(CD, "\nParte 5. Combos do Strat no diário (versões de alta, 40 ações desde 2005; resultado em R, sem custos)")
print(f"   barras que abriram acima do gatilho (entrada na abertura): {pc(CD['gap'].mean())} dos casos")

In [ ]:
import matplotlib.pyplot as plt
nomes = [k for k in ORDEM if (CD["combo"] == k).any()]
g = CD.groupby("combo")[["r", "r_otim"]].mean().reindex(nomes)
ax = g.plot(kind="barh", figsize=(9, 4), color=["black", "white"], edgecolor="black",
            title="Combos no diário: resultado médio em R (conservador x otimista)")
ax.axvline(0, color="gray")
ax.invert_yaxis()
plt.show()

## Parte 6: magnitude

In [ ]:
print("\nParte 6. Magnitude: os mesmos combos (sem o 2-2 de continuação, cujo alvo é sempre 1 R), pela distância até o alvo")
M = CD[CD["combo"] != "2-2 de continuação"].copy()
M["faixa"] = pd.cut(M["dist"], [0, 0.5, 1.0, np.inf], labels=["alvo a menos de 0,5 R", "de 0,5 a 1 R", "1 R ou mais"])
for k, x in M.groupby("faixa", observed=True):
    print(f"   {k:<22} {nn(len(x)):>7} casos | {nr(x['r'].mean())} R (otimista {nr(x['r_otim'].mean())} R) |"
          f" acerto {pc((x['r'] > 0).mean())} ({pc((x['r_otim'] > 0).mean())})")

## Parte 7: os combos no 60 minutos, com e sem continuidade temporal

In [ ]:
def continuidade(x):
    """Preço no fechamento de cada barra de 60 minutos comparado com a abertura do dia, da semana e do mês."""
    dia = x.index.tz_localize(None).normalize() if x.index.tz is not None else x.index.normalize()
    sem = dia.to_period("W-FRI")
    mes = dia.to_period("M")
    ab_dia = x["Open"].groupby(dia).transform("first")
    ab_sem = x["Open"].groupby(sem).transform("first")
    ab_mes = x["Open"].groupby(mes).transform("first")
    c = x["Close"]
    verde = (c > ab_dia) & (c > ab_sem) & (c > ab_mes)
    vermelho = (c < ab_dia) & (c < ab_sem) & (c < ab_mes)
    estado = pd.Series(np.select([verde, vermelho], ["todos verdes", "todos vermelhos"], "misturado"), index=x.index)
    estado[mes == mes[0]] = ""                                # o primeiro mês da amostra começa pela metade
    return estado


C60 = []
for t in INTRA:
    x = limpa(H60[t])
    e = continuidade(x).to_numpy()
    c = combos(x, prazo=14)                                   # 14 barras: cerca de dois pregões
    c["tfc"] = e[c["i"].to_numpy() - 1]                       # estado no fechamento da barra anterior ao gatilho
    C60.append(c.assign(ativo=t))
C60 = pd.concat(C60).dropna(subset=["r", "r_otim"])
C60 = C60[C60["tfc"] != ""]
tabela(C60, "\nParte 7. Combos no gráfico de 60 minutos (8 ativos, ~2 anos; prazo de 14 barras)")
print("   Os mesmos combos, pelo estado da continuidade (dia, semana e mês) antes do gatilho:")
for k, x in C60.groupby("tfc"):
    print(f"   {k:<16} {nn(len(x)):>6} casos ({pc(len(x) / len(C60))}) | {nr(x['r'].mean())} R (otimista {nr(x['r_otim'].mean())} R)"
          f" | acerto {pc((x['r'] > 0).mean())}")
amb = (C60["r"] != C60["r_otim"]).mean()
amb_d = (CD["r"] != CD["r_otim"]).mean()
print(f"   casos em que a ordem dentro da barra muda o resultado: diário {pc(amb_d)} | 60 minutos {pc(amb)}")

## Parte 8: continuidade temporal sozinha

In [ ]:
print("\nParte 8. Continuidade temporal sozinha")
T = []
for tk in TICKERS:
    if tk not in SERIES:
        continue
    d = SERIES[tk]
    px = d["Close"]
    ab_mes = d["Open"].groupby(d.index.to_period("M")).transform("first")
    ab_sem = d["Open"].groupby(d.index.to_period("W-FRI")).transform("first")
    verde = (px > ab_mes) & (px > ab_sem) & (px > d["Open"])
    vermelho = (px < ab_mes) & (px < ab_sem) & (px < d["Open"])
    T.append(pd.DataFrame({"estado": np.select([verde, vermelho], ["todos verdes", "todos vermelhos"], "misturado"),
                           "f": px.shift(-5) / px - 1}))
T = pd.concat(T).dropna()
print("   Diário, 40 ações (mês, semana e dia; retorno nos 5 pregões seguintes):")
for k, x in T.groupby("estado"):
    print(f"   {k:<16} {pc(len(x) / len(T)):>4} dos dias | média {ps(x['f'].mean())} | positivo em {pc((x['f'] > 0).mean())}")
T6 = []
for t in INTRA:
    x = limpa(H60[t])
    e = continuidade(x)
    T6.append(pd.DataFrame({"estado": e, "f": x["Close"].shift(-7) / x["Close"] - 1}))
T6 = pd.concat(T6).dropna()
T6 = T6[T6["estado"] != ""]
print("   60 minutos, 8 ativos (dia, semana e mês; retorno nas 7 barras seguintes, cerca de um pregão):")
for k, x in T6.groupby("estado"):
    print(f"   {k:<16} {pc(len(x) / len(T6)):>4} das barras | média {ps(x['f'].mean(), 3)} | positivo em {pc((x['f'] > 0).mean())}")

## Parte 9: o relógio muda o gráfico

In [ ]:
d = IX["Ibovespa"]
seg = cenario(agrega(d, "W-FRI")).iloc[1:]                # semana de segunda a sexta
qua = cenario(agrega(d, "W-TUE")).iloc[1:]                # "semana" de quarta a terça
print("\nParte 9. Ibovespa semanal: semana de segunda a sexta x semana de quarta a terça")
for nome, s in [("segunda a sexta", seg), ("quarta a terça", qua)]:
    v = s.str[0].value_counts(normalize=True)
    print(f"   {nome:<16} 1: {pc(v.get('1', 0))} | 2: {pc(v.get('2', 0))} | 3: {pc(v.get('3', 0))}")
print("   A proporção é parecida, mas as barras 3 caem em semanas diferentes: o que se vê depende de quando a barra começa.")

## Parte 10: exemplo real (Ibovespa)

In [ ]:
m = agrega(d, "ME")
cm = cenario(m)
ult = cm[cm == "3"].index[-1]
j = m.index.get_loc(ult)
print("\nParte 10. A última barra 3 do Ibovespa mensal (para as figuras)")
print(f"   {ult:%m/%Y}: máxima {nr(m['High'].iloc[j], 0)[1:]} e mínima {nr(m['Low'].iloc[j], 0)[1:]};"
      f" mês anterior: máxima {nr(m['High'].iloc[j - 1], 0)[1:]} e mínima {nr(m['Low'].iloc[j - 1], 0)[1:]}")
dd = d[(d.index >= m.index[j - 1] - pd.offsets.MonthBegin(1)) & (d.index <= ult)]
print(f"   no diário, de {dd.index[0]:%d/%m/%Y} a {dd.index[-1]:%d/%m/%Y}: {len(dd)} pregões;"
      f" mínima em {dd['Low'].idxmin():%d/%m/%Y} e máxima em {dd['High'].idxmax():%d/%m/%Y}")
for k in range(j + 1, min(j + 4, len(m))):
    print(f"   {m.index[k]:%m/%Y}: cenário {cm.iloc[k]} | fechamento {nr(m['Close'].iloc[k], 0)[1:]}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `cenario` da Parte 1. Mostre o que fazem `h.shift()` e `l.shift()`, por que o número depende só da máxima e da mínima e por que a primeira barra fica sem cenário." Confira com a Parte 2, cuja última coluna conta o "2 que fechou contra" (de 20% a 25% das barras 2, no livro): a cor da barra não muda o cenário.
2. **Adaptar:** "Numa célula nova depois da Parte 2, baixe o diário de [ATIVO] do Yahoo Finance desde 2000, passe por `limpa` e conte as barras 1, 2 e 3 com `cenario` no diário, no semanal (`agrega(d, "W-FRI")`) e no mensal (`agrega(d, "ME")`), como na Parte 2. Não mude `cenario`." Antes de olhar o ativo novo, confira se a Parte 2 mostra os números da Tabela 24.1: no Ibovespa diário, 12% de barras 1, 80% de barras 2 e 7% de barras 3.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 5, troque o alvo pela máxima anterior por um alvo fixo de 2R. O que acontece com o acerto e com o resultado?
2. Na Parte 6, opere só os combos com alvo a 1 R ou mais e com continuidade verde. Quantos casos sobram?
3. Na Parte 7, inclua a abertura da hora na continuidade (quatro tempos, como Smith). O filtro melhora?
4. Na Parte 9, monte o semanal começando na quinta-feira. As barras 3 mudam de lugar?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.